# Extract Data from Data Knowledge Graph

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from neo4j import GraphDatabase, RoutingControl, Result

In [2]:
# load secrets from .env file
load_dotenv()
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")

In [ ]:
# Define and check connection of DKG
url = url # your Neo4j database URL
user = db_user # your username
password = db_password # your password
db_name = db_name # your database name
driver = GraphDatabase.driver(url, auth=(user, password))

driver.verify_connectivity()

In [ ]:
# get assets - business terms, from the graph and store them in a dataframe
with driver.session() as session:
    query = (
            '''
            MATCH (n:Asset)
            WHERE n.type_name = 'business_term'
            RETURN n.id as id, n.asset_source_id as source_id, n.asset_source_url as source_url, 
                n.name as name, n.name_embedding as name_embedding, n.definition as definition;
            '''
    )
    result = session.run(query)
    term_data = pd.DataFrame([dict(record) for record in result])

In [ ]:
# get other string attributes of business terms
with driver.session() as session:
    query = (
            '''
            MATCH (n:Asset)-[r]-(t:Attribute)
            WHERE n.type_name = 'business_term' AND t.type_name in ['note', 'definition', 'descriptive_example']
            RETURN t.asset_id as AssetId, t.type_name as AttributeType, t.value as AttributeValue;
            '''
    )
    result = session.run(query)
    other_attributes = pd.DataFrame([dict(record) for record in result])